# LightAD full benchmark on Kaggle

Notebook chạy LightAD DT/KNN/SLFN trên full HDFS_v1 và BGL.

## Kaggle datasets cần Add data
- HDFS_v1: `HDFS.log`, `preprocessed/Event_occurrence_matrix.csv`, `preprocessed/anomaly_label.csv`
- BGL: `BGL.log`

Kết quả được lưu trong `/kaggle/working`.

In [ ]:
from pathlib import Path
from collections import Counter
import json, time, platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_recall_fscore_support, average_precision_score

SEEDS = [42, 123, 456, 789, 2024]
MODELS = ["dt", "knn", "slfn"]
WINDOW_SIZE = 200
WORKING = Path('/kaggle/working' if Path('/kaggle/working').exists() else './results')
WORKING.mkdir(parents=True, exist_ok=True)
print('Platform:', platform.platform())
print('Working:', WORKING)

## 1. Tìm dữ liệu

In [ ]:
def find_file(filename):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.')]
    for root in roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    raise FileNotFoundError(filename)

hdfs_matrix = find_file('Event_occurrence_matrix.csv')
hdfs_labels = find_file('anomaly_label.csv')
bgl_log = find_file('BGL.log')
hdfs_raw = find_file('HDFS.log')

print('HDFS matrix:', hdfs_matrix)
print('HDFS labels:', hdfs_labels)
print('HDFS raw:', hdfs_raw)
print('BGL raw:', bgl_log)

## 2. HDFS_v1 full

In [ ]:
hdfs = pd.read_csv(hdfs_matrix)
feature_columns = [f'E{i}' for i in range(1, 30)]
missing = [column for column in feature_columns if column not in hdfs.columns]
if missing:
    raise ValueError(f'Missing HDFS columns: {missing}')

X_hdfs = hdfs[feature_columns].to_numpy(dtype=np.float32)
y_hdfs = hdfs['Label'].replace({
    'Normal': 0, 'Success': 0, 'Anomaly': 1, 'Fail': 1
}).to_numpy(dtype=np.int64)

print('HDFS shape:', X_hdfs.shape)
print('HDFS labels:', Counter(y_hdfs.tolist()))

## 3. Parse BGL và gắn nhãn

Cột đầu tiên là alert label: `-` là Normal, category khác `-` là Anomaly.

In [ ]:
def parse_bgl(raw_path):
    try:
        from drain3 import TemplateMiner
        from drain3.template_miner_config import TemplateMinerConfig
    except ImportError:
        get_ipython().system('pip install -q drain3==0.9.11')
        from drain3 import TemplateMiner
        from drain3.template_miner_config import TemplateMinerConfig

    config = TemplateMinerConfig()
    config.profiling_enabled = False
    miner = TemplateMiner(config=config)
    rows = []
    with open(raw_path, 'r', encoding='utf-8', errors='replace') as stream:
        for line_id, line in enumerate(stream, start=1):
            raw = line.rstrip('\r\n')
            if not raw:
                continue
            first = raw.split(maxsplit=1)[0]
            result = miner.add_log_message(raw)
            rows.append({
                'template_id': str(result['cluster_id']),
                'label': 'Normal' if first == '-' else 'Anomaly',
            })
    return pd.DataFrame(rows)

bgl_cache = WORKING / 'bgl_structured.csv'
if bgl_cache.exists():
    bgl = pd.read_csv(bgl_cache)
    print('Loaded cache:', bgl_cache)
else:
    bgl = parse_bgl(bgl_log)
    bgl.to_csv(bgl_cache, index=False)
    print('Saved:', bgl_cache)

print('BGL shape:', bgl.shape)
print('BGL labels:', Counter(bgl['label'].tolist()))
print('Unique templates:', bgl['template_id'].nunique())

In [ ]:
def build_bgl_windows(bgl, window_size=200):
    ordered = sorted(bgl['template_id'].astype(str).unique(),
                      key=lambda value: int(value) if value.isdigit() else 10**9)
    index = {template_id: i for i, template_id in enumerate(ordered)}
    x_rows, y_rows = [], []
    counts = Counter()
    window_anomaly = 0
    position = 0

    for row in bgl.itertuples(index=False):
        counts[index[str(row.template_id)]] += 1
        if row.label == 'Anomaly':
            window_anomaly = 1
        position += 1
        if position % window_size == 0:
            x_rows.append([counts[i] for i in range(len(ordered))])
            y_rows.append(window_anomaly)
            counts = Counter()
            window_anomaly = 0

    return np.asarray(x_rows, dtype=np.float32), np.asarray(y_rows, dtype=np.int64), ordered

X_bgl, y_bgl, bgl_features = build_bgl_windows(bgl, WINDOW_SIZE)
print('BGL window shape:', X_bgl.shape)
print('BGL window labels:', Counter(y_bgl.tolist()))

## 4. Split và metric

In [ ]:
def hdfs_split(y, seed):
    indices = np.arange(len(y))
    train_idx, temp_idx = train_test_split(indices, test_size=0.20,
                                           random_state=seed, stratify=y)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50,
                                         random_state=seed, stratify=y[temp_idx])
    return train_idx, val_idx, test_idx

def chronological_split(y):
    train_end = int(0.8 * len(y))
    val_end = int(0.9 * len(y))
    return np.arange(train_end), np.arange(train_end, val_end), np.arange(val_end, len(y))

def make_model(name, seed):
    if name == 'dt':
        return DecisionTreeClassifier(random_state=seed)
    if name == 'knn':
        return KNeighborsClassifier(n_neighbors=1, n_jobs=-1)
    if name == 'slfn':
        return MLPClassifier(hidden_layer_sizes=(25,), max_iter=300, random_state=seed)
    raise ValueError(name)

def compute_metrics(y_true, y_pred, y_score=None):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0
    )
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fpr = fp / (fp + tn) if fp + tn else 0.0
    pr_auc = None
    if y_score is not None and len(np.unique(y_true)) == 2:
        pr_auc = float(average_precision_score(y_true, y_score))
    return {'precision': float(precision), 'recall': float(recall),
            'f1': float(f1), 'fpr': float(fpr), 'pr_auc': pr_auc,
            'tn': tn, 'fp': fp}

In [ ]:
def run_dataset(X, y, split_kind):
    all_runs = {model_name: [] for model_name in MODELS}
    for seed in SEEDS:
        if split_kind == 'stratified':
            train_idx, val_idx, test_idx = hdfs_split(y, seed)
        else:
            train_idx, val_idx, test_idx = chronological_split(y)

        for model_name in MODELS:
            model = make_model(model_name, seed)
            start = time.perf_counter()
            model.fit(X[train_idx], y[train_idx])
            train_time = time.perf_counter() - start

            start = time.perf_counter()
            y_pred = model.predict(X[test_idx])
            infer_time = time.perf_counter() - start
            y_score = model.predict_proba(X[test_idx])[:, 1] if hasattr(model, 'predict_proba') else None

            metrics = compute_metrics(y[test_idx], y_pred, y_score)
            metrics.update({
                'seed': seed,
                'train_time_seconds': train_time,
                'inference_time_seconds': infer_time,
                'latency_per_sample_ms': infer_time / len(test_idx) * 1000,
                'test_samples': len(test_idx),
                'train_samples': len(train_idx),
            })
            all_runs[model_name].append(metrics)

    summarized = {}
    for model_name, runs in all_runs.items():
        summarized[model_name] = {}
        for metric in ['precision', 'recall', 'f1', 'pr_auc', 'fpr']:
            values = [0.0 if run[metric] is None else run[metric] for run in runs]
            summarized[model_name][metric] = {
                'mean': float(np.mean(values)),
                'std': float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
            }
        summarized[model_name]['train_time_seconds'] = {
            'mean': float(np.mean([run['train_time_seconds'] for run in runs]))
        }
        summarized[model_name]['inference_time_seconds'] = {
            'mean': float(np.mean([run['inference_time_seconds'] for run in runs]))
        }
    return {'summary': summarized, 'runs': all_runs}

In [ ]:
print('Running HDFS_v1 full dataset...')
hdfs_results = run_dataset(X_hdfs, y_hdfs, 'stratified')
print(json.dumps(hdfs_results['summary'], indent=2))

In [ ]:
print('Running BGL full dataset in 200-line windows...')
bgl_results = run_dataset(X_bgl, y_bgl, 'chronological')
print(json.dumps(bgl_results['summary'], indent=2))

## 5. Lưu kết quả

In [ ]:
final_results = {
    'hdfs_v1': hdfs_results,
    'bgl': bgl_results,
    'seeds': SEEDS,
    'bgl_window_size': WINDOW_SIZE,
}
json_path = WORKING / 'lightad_full_benchmark_results.json'
json_path.write_text(json.dumps(final_results, indent=2, ensure_ascii=False), encoding='utf-8')

rows = []
for dataset_name, result in [('HDFS_v1', hdfs_results), ('BGL', bgl_results)]:
    for model_name, metrics in result['summary'].items():
        rows.append({
            'Dataset': dataset_name,
            'Model': model_name.upper(),
            'Precision_mean': metrics['precision']['mean'],
            'Precision_std': metrics['precision']['std'],
            'Recall_mean': metrics['recall']['mean'],
            'Recall_std': metrics['recall']['std'],
            'F1_mean': metrics['f1']['mean'],
            'F1_std': metrics['f1']['std'],
            'PR_AUC_mean': metrics['pr_auc']['mean'],
            'PR_AUC_std': metrics['pr_auc']['std'],
            'FPR_mean': metrics['fpr']['mean'],
            'FPR_std': metrics['fpr']['std'],
        })
table = pd.DataFrame(rows)
csv_path = WORKING / 'lightad_full_benchmark_results.csv'
table.to_csv(csv_path, index=False)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dataset_name in zip(axes, ['HDFS_v1', 'BGL']):
    subset = table[table['Dataset'] == dataset_name]
    ax.bar(subset['Model'], subset['F1_mean'], yerr=subset['F1_std'], capsize=4)
    ax.set_title(dataset_name)
    ax.set_ylabel('F1')
    ax.set_ylim(0, 1.05)
plt.tight_layout()
png_path = WORKING / 'lightad_f1_comparison.png'
fig.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', json_path)
print('Saved:', csv_path)
print('Saved:', png_path)

## 6. Ghi chú

- LogFormer và LogRAG chưa chạy trong notebook này.
- Cần pretrained checkpoint/model hoặc LLM backend trước khi benchmark các phương pháp đó.
- Tải JSON/CSV/PNG trong phần Output của Kaggle.